In [1]:
from gensim.models import Word2Vec


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [3]:
import pandas as pd
import numpy as np

Maybe we should use grams model, but i am not sure about it.

In [4]:
data=[]
for element in pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\ChunckByte.csv")["content"]:
    data.append(element.split(" "))

<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
C:\Users\hsfac\AppData\Local\Temp\ipykernel_24772\3937927017.py:2: SyntaxWarning: invalid escape sequence '\S'
  for element in pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\ChunckByte.csv")["content"]:


In [5]:
model1 = Word2Vec(data, min_count=1,
                                vector_size=128, window=5)

In [6]:
def convert_to_list_vector(data):
   vector_list = [model1.wv[word] for word in data]
   #print(type(vector_list[0]))
   #word_matrix = np.array(vector_list)
   #print(word_matrix.shape)
   return torch.tensor(vector_list)

In [7]:
max=0
t=[]
for element in pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\ChunckByte.csv")["content"]:
    if max>len(element.split(" ")):
        pass
    else:
        t=element.split(" ")
        max=len(element.split(" "))


<>:3: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
C:\Users\hsfac\AppData\Local\Temp\ipykernel_24772\1117437580.py:3: SyntaxWarning: invalid escape sequence '\S'
  for element in pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\ChunckByte.csv")["content"]:


In [120]:
"""class WordRNN(nn.Module):
    def __init__(self,input_size,hidden_size,layer_size,output_size=1):
        super(WordRNN, self).__init__()
        self.hidden_size=hidden_size
        self.layer_size=layer_size
        self.lstm = nn.LSTM(input_size, hidden_size,layer_size, batch_first=True)
        # A standard Linear layer to get the final prediction
        self.fc = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=1)
    def forward(self,x):
        #h0 = torch.zeros(self.layer_size, x.size(), self.hidden_size)
        #c0 = torch.zeros(self.layer_size, x.size(), self.hidden_size)
        out, _ = self.lstm(x)
        #out = self.fc(out[:, -1, :])
        
        # 4. Apply Softmax (if doing classification)
        #out = self.softmax(out)
        return out
"""
class WordRNN(nn.Module):
    def __init__(self,embedding_dim, hidden_dim, vocab_size, tagset_size=1):
        super(WordRNN, self).__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim,batch_first=True)
        self.fc=nn.Linear(hidden_dim,1)
        #self.fc2=nn.Linear(32,1)

        # The linear layer that maps from hidden state space to tag space
        #self.hidden2tag = nn.Linear(hidden_dim, tagset_size)
    def forward(self,sentence):
        output,(h_n,c_n)=self.lstm(sentence)
        #h_n=self.fc(h_n)
        last_hidden=h_n[-1]
        
        #last_hidden=torch.sigmoid(last_hidden)
        logit=self.fc(last_hidden)
        logit=torch.sigmoid(logit)
        #embeds = self.word_embeddings(sentence)
        #lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        #tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        #tag_scores = F.log_softmax(tag_space, dim=1)
        return logit

In [11]:
pd_data=pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\ChunckByte.csv")
pd_data["type"]=pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\linear_regreasion\995,000_rows_stemming_words.csv",nrows=50000)["type"]

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
C:\Users\hsfac\AppData\Local\Temp\ipykernel_24772\2594029508.py:1: SyntaxWarning: invalid escape sequence '\S'
  pd_data=pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\ChunckByte.csv")
C:\Users\hsfac\AppData\Local\Temp\ipykernel_24772\2594029508.py:2: SyntaxWarning: invalid escape sequence '\S'
  pd_data["type"]=pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\linear_regreasion\995,000_rows_stemming_words.csv",nrows=50000)["type"]
C:\Users\hsfac\AppData\Local\Temp\ipykernel_24772\2594029508.py:2: DtypeWarning: Columns (0: Unnamed: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_data["type"]=pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\linear_regreasion\995,000_rows

In [57]:
print(pd_data["type"][10])

political


In [94]:
t=pd_data["content"][10].split(" ")
convert_to_list_vector(t).shape

torch.Size([271, 128])

In [158]:
traning_data=[]
traning_data.append((convert_to_list_vector(pd_data["content"][1].split(" ")),0))
traning_data.append((convert_to_list_vector(pd_data["content"][3].split(" ")),1))

In [159]:
model = WordRNN(128, 256, 2)

In [160]:
model(convert_to_list_vector(t))

tensor([0.4777], grad_fn=<SigmoidBackward0>)

In [194]:
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [180]:
import math

In [195]:
for epoch in range(100):
    for arg,ty in traning_data:
        optimizer.zero_grad()
        outputs=model(arg)
        loss=criterion(outputs,torch.Tensor([ty]))
        loss.backward()
        optimizer.step()

In [221]:
for arg, ty in traning_data:
    print(len(arg))

133
591


In [220]:
model.eval()
with torch.no_grad():
    outputs=model(convert_to_list_vector(pd_data["content"][431].split(" ")))
    prediction=torch.max(outputs)
    print(prediction)

tensor(4.5736e-07)


In [ ]:
model = WordRNN(6, 6, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# See what the scores are before training
# Note that element i,j of the output is the score for tag j for word i.
# Here we don't need to train, so the code is wrapped in torch.no_grad()
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)
    print(tag_scores)

for epoch in range(300):  # again, normally you would NOT do 300 epochs, it is toy data
    for sentence, tags in training_data:
        # Step 1. Remember that Pytorch accumulates gradients.
        # We need to clear them out before each instance
        model.zero_grad()

        # Step 2. Get our inputs ready for the network, that is, turn them into
        # Tensors of word indices.
        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)

        # Step 3. Run our forward pass.
        tag_scores = model(sentence_in)

        # Step 4. Compute the loss, gradients, and update the parameters by
        #  calling optimizer.step()
        loss = loss_function(tag_scores, targets)
        loss.backward()
        optimizer.step()

# See what the scores are after training
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)

    # The sentence is "the dog ate the apple".  i,j corresponds to score for tag j
    # for word i. The predicted tag is the maximum scoring tag.
    # Here, we can see the predicted sequence below is 0 1 2 0 1
    # since 0 is index of the maximum value of row 1,
    # 1 is the index of maximum value of row 2, etc.
    # Which is DET NOUN VERB DET NOUN, the correct sequence!
    print(tag_scores)

In [9]:
epochs=10
model=WordRNN(128,256,2)
lr=0.001


In [10]:
#device = torch.device('cuda', infected if torch.cuda.is_available() else 'cpu')
model = WordRNN(128, 256, 2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
pd_data=pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\ChunckByte.csv")
pd_data["type"]=pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\linear_regreasion\995,000_rows_stemming_words.csv",nrows=50000)["type"]

#print(pd_data["content"][0])

for epoch in range(epochs):
    for index,element in pd_data[:4].iterrows():
        articale=element["content"].split(" ")
        label=element["type"]
        articale=convert_to_list_vector(articale)
        outputs = model(articale)
        loss = criterion(outputs, label)
        
        # Backward and optimize
        optimizer.zero_grad()
        #loss.backward()
        
        # Gradient clipping (prevents the model from "exploding")
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        
        optimizer.step()
        
        #total_loss += loss.item()
        
    #print(f'Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(len(pd_data)):.4f}')

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:2: SyntaxWarning: invalid escape sequence '\S'
C:\Users\hsfac\AppData\Local\Temp\ipykernel_28932\581042570.py:1: SyntaxWarning: invalid escape sequence '\S'
  pd_data=pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\ChunckByte.csv")
C:\Users\hsfac\AppData\Local\Temp\ipykernel_28932\581042570.py:2: SyntaxWarning: invalid escape sequence '\S'
  pd_data["type"]=pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\linear_regreasion\995,000_rows_stemming_words.csv",nrows=50000)["type"]
C:\Users\hsfac\AppData\Local\Temp\ipykernel_28932\581042570.py:2: DtypeWarning: Columns (0: Unnamed: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  pd_data["type"]=pd.read_csv("C:\Study_Machine_Learning_ku\GDS\Project\Group-Fake-News-Project\linear_regreasion\995,000_rows_st